# Lab Evaluate Your RAG: Test Set, Retrieval Metrics, Faithfulness

**AI Agents course Day 2 (RAG: retrieval, embeddings and evaluation)**

Day2_RAG_Pipeline built a pipeline and checked it on 14 questions. Here you turn that check into an **evaluation**: a
test set you can trust, retrieval measured on its own (hit rate, MRR) for three retrievers, answers checked for
correctness and **faithfulness**, a judge you check against your own labels -- and an honest error bar.

Same documents: 38 pages of GitLab's public handbook at a pinned commit (CC BY-SA 4.0).


### Learning objectives

- **Build** a test set: hand-written questions, LLM-drafted questions checked against the source, and questions the
  documents cannot answer.
- **Measure** retrieval separately with hit@k and MRR, and compare keyword, vector and hybrid search.
- **Check** answers with rules where you can and an LLM judge where you must -- and **check the judge** against your
  own labels.
- **Diagnose** failures with the retrieval/generation 2×2 and report results with error bars.
- *Extras, if you finish early:* are generated questions easier? Change one thing and measure again.

### Setup

Runtime → Change runtime type → **T4 GPU**, then run the setup cells. Loading the model takes a few minutes.

In [2]:
#@title Install libraries
!pip install -q -U transformers accelerate sentence-transformers rank_bm25

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 12.8/12.8 MB 111.1 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 740.6/740.6 kB 58.1 MB/s eta 0:00:00


In [3]:
#@title Load Qwen3.5-4B (the same model in every lab)
import torch
from transformers import AutoTokenizer, AutoModelForImageTextToText

MODEL_ID = "Qwen/Qwen3.5-4B"                     # about 9 GB in float16: fits a free Colab T4
tok = AutoTokenizer.from_pretrained(MODEL_ID)
model = AutoModelForImageTextToText.from_pretrained(
    MODEL_ID, dtype=torch.float16 if torch.cuda.is_available() else torch.float32, device_map="auto")
model.eval()
print("running on:", torch.cuda.get_device_name(0) if torch.cuda.is_available() else "CPU")

config.json:   0%|          | 0.00/3.16k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/16.7k [00:00<?, ?B/s]

tokenizer.json: reconstructing file:   0%|          |  0.00B / 12.8MB            

tokenizer.json: downloading bytes:           |  0.00B            

chat_template.jinja:   0%|          | 0.00/7.76k [00:00<?, ?B/s]

model.safetensors.index.json:   0%|          | 0.00/76.2k [00:00<?, ?B/s]

Reconstructing (incomplete total...): |          |  0.00B /  0.00B            

Fetching 2 files:   0%|          | 0/2 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/723 [00:00<?, ?it/s]

running on: Tesla T4


In [4]:
#@title chat(): the one function that talks to the model (the same in every lab)
def chat(messages, system=None, max_new_tokens=400, temperature=0.0, thinking=False, tools=None):
    """A prompt string or a list of {"role", "content"} messages -> the model's reply as text.
    system: an optional system prompt, put in front of the messages.
    temperature=0 is greedy (the same reply every time); above 0 it samples, with top_p=0.8 and top_k=20
    from the Qwen3.5 model card. thinking=True lets the model reason in <think>...</think> before answering.
    tools: tool schemas for native tool calling (the reply then may contain <tool_call> blocks)."""
    if isinstance(messages, str):
        messages = [{"role": "user", "content": messages}]
    if system:
        messages = [{"role": "system", "content": system}] + messages
    prompt = tok.apply_chat_template(messages, tools=tools, tokenize=False, add_generation_prompt=True,
                                     enable_thinking=thinking)                 # wrap in the model's own chat markup
    inputs = tok(prompt, return_tensors="pt").to(model.device)
    sampling = dict(do_sample=True, temperature=temperature, top_p=0.8, top_k=20) if temperature > 0 else dict(do_sample=False)
    with torch.no_grad():
        out = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **sampling)
    return tok.decode(out[0][inputs["input_ids"].shape[1]:], skip_special_tokens=True).strip()  # keep only the new text

In [5]:
#@title chat_batch(): chat() for many prompts at once
def chat_batch(prompts, system=None, max_new_tokens=400, temperature=0.0, thinking=False, batch_size=16):
    """Same settings as chat(), but one generate() call per batch_size prompts: much faster on a GPU."""
    conversations = [[{"role": "user", "content": p}] if isinstance(p, str) else p for p in prompts]
    if system:
        conversations = [[{"role": "system", "content": system}] + c for c in conversations]
    texts = [tok.apply_chat_template(c, tokenize=False, add_generation_prompt=True, enable_thinking=thinking)
             for c in conversations]
    tok.padding_side = "left"                     # pad on the left, so every reply starts right after its prompt
    sampling = dict(do_sample=True, temperature=temperature, top_p=0.8, top_k=20) if temperature > 0 else dict(do_sample=False)
    replies = []
    for i in range(0, len(texts), batch_size):
        inputs = tok(texts[i:i + batch_size], return_tensors="pt", padding=True).to(model.device)
        with torch.no_grad():
            out = model.generate(**inputs, max_new_tokens=max_new_tokens, pad_token_id=tok.pad_token_id, **sampling)
        replies += tok.batch_decode(out[:, inputs["input_ids"].shape[1]:], skip_special_tokens=True)
    return [r.strip() for r in replies]

In [6]:
#@title Imports for this lab, and the embedding model
import re, json, time, random, collections
import numpy as np, pandas as pd
from sentence_transformers import SentenceTransformer
pd.set_option("display.max_colwidth", None)                  # show whole questions and answers in tables

embedder = SentenceTransformer("all-MiniLM-L6-v2", device="cuda" if torch.cuda.is_available() else "cpu")

modules.json:   0%|          | 0.00/349 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/116 [00:00<?, ?B/s]

README.md:   0%|          | 0.00/10.5k [00:00<?, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/53.0 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/612 [00:00<?, ?B/s]

...

tokenizer_config.json:   0%|          | 0.00/350 [00:00<?, ?B/s]

vocab.txt:   0%|          | 0.00/232k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/466k [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/112 [00:00<?, ?B/s]

config.json:   0%|          | 0.00/190 [00:00<?, ?B/s]

In [7]:
#@title Fetch the handbook (same pinned commit as Day2_RAG_Pipeline) and chunk it
import requests, concurrent.futures as cf
from urllib.parse import quote

API = "https://gitlab.com/api/v4/projects/gitlab-com%2Fcontent-sites%2Fhandbook/repository"
REF = "df66e66b937d38c1ed4e3dd452927ddf01be58b0"
PAGES = '''people-group/time-off-and-absence/time-off-types.md people-group/time-off-and-absence/leave-types.md
people-group/anti-harassment.md people-group/contracts-probation-periods.md people-group/acceptable-use-policy.md
people-group/general-onboarding/_index.md people-group/offboarding/faq.md people-policies/inc-usa.md
people-policies/leave-of-absence/us.md people-policies/ireland-ltd/right-to-disconnect-policy.md
people-policies/france-sas/remote-work-charter.md total-rewards/benefits/_index.md
total-rewards/benefits/parental-leave-toolkit.md total-rewards/benefits/general-and-entity-benefits/ltd-benefits-uk.md
total-rewards/benefits/general-and-entity-benefits/gitlab-ireland-ltd.md
total-rewards/benefits/general-and-entity-benefits/bv-benefits-netherlands.md
total-rewards/benefits/general-and-entity-benefits/inc-benefits-us/_index.md
total-rewards/benefits/general-and-entity-benefits/pty-benefits-australia.md
total-rewards/benefits/general-and-entity-benefits/canada-corp-benefits.md total-rewards/compensation/_index.md
total-rewards/stock-options.md finance/expenses.md finance/spending-company-money.md
finance/procurement/individual-use-software.md finance/accounts-payable/corp-credit-cards.md
security/policies_and_standards/password-standard.md security/policies_and_standards/data-classification-standard.md
security/policies_and_standards/physical-security-standard-for-company-assets.md
security/policies_and_standards/records-retention-deletion.md security/policies_and_standards/token-management-standard.md
legal/anti-corruption-policy.md legal/anti-fraud-policy.md legal/anti-retaliation-policy.md
legal/policies/gifts-contributions.md legal/acceptable-use-policy.md business-technology/_index.md
communication/confidentiality-levels.md hiring/referral-process.md'''.split()

def fetch(page):
    raw = requests.get(f"{API}/files/{quote('content/handbook/' + page, safe='')}/raw", params={"ref": REF}).text
    front, body = raw.split("\n---\n", 1)
    body = re.sub(r"\{\{[<%].*?[>%]\}\}|<!--.*?-->|<[^>\n]+>", "", body, flags=re.S)
    body = re.sub(r"\[([^\]]+)\]\([^)]*\)", r"\1", body)
    return {"title": re.search(r'^title:\s*"?(.*?)"?\s*$', front, re.M).group(1), "text": re.sub(r"[ \t]+", " ", body)}

with cf.ThreadPoolExecutor(8) as pool:
    docs = list(pool.map(fetch, PAGES))

def chunk_fixed(text, size=300):
    pieces, current = [], ""
    for word in text.split():
        if len(current) + len(word) > size:
            pieces.append(current.strip()); current = ""
        current += word + " "
    return pieces + [current.strip()]

chunks = [{"title": d["title"], "text": p} for d in docs for p in chunk_fixed(d["text"], 300) if len(p) > 40]
print(len(docs), "pages,", len(chunks), "chunks of about 300 characters")

38 pages, 2108 chunks of about 300 characters


In [8]:
#@title Two helpers used everywhere (read once)
def contains(text, answer):
    "Does `text` contain `answer` as whole words (ignoring case)? '4%' is found in 'pays 4% of' but not in '0.54%'."
    return re.search(r"(?<![\w.])" + re.escape(answer.lower()) + r"(?!\w)", text.lower()) is not None

def fill(template, **values):
    "Put values into {name} placeholders. Unlike str.format, it leaves JSON braces such as {\"question\": ...} alone."
    return re.sub(r"\{(\w+)\}", lambda m: str(values[m.group(1)]) if m.group(1) in values else m.group(0), template)

assert contains("GitLab contributes 4% of your basic salary", "4%") and not contains("Annual Management Charge: 0.54%", "4%")

## 1 The test set

A test set has three kinds of rows. Each answerable question carries the **answer text** that must appear in the
retrieved context -- that is what makes retrieval checkable without a model.

**Method 1 -- written by a person.** The 14 questions from Day2_RAG_Pipeline.

In [9]:
HAND = [
    ("How many days of time off should I take at minimum each year?", "25 days"),
    ("What percentage of my salary does the company pay into the UK pension?", "4%"),
    ("Which private medical insurance provider covers team members in Ireland?", "VHI"),
    ("What is the daily limit for meals while traveling for work?", "$120"),
    ("How many weeks of paid parental leave do team members get?", "16 weeks"),
    ("What is the minimum password length?", "12 characters"),
    ("How long can a personal access token be valid before it expires?", "30 days"),
    ("Can I install Chrome extensions on my work laptop?", "Chrome Extensions"),
    ("What is the maximum value of a gift to a US government official per event?", "$20"),
    ("How do I return my laptop when I leave?", "prepaid shipping label"),
    ("How long are production change records retained?", "3 years"),
    ("Are interns eligible for referral bonuses?", "not eligible"),
    ("How many vacation days are Netherlands team members entitled to?", "20 vacation days"),
    ("What is the reimbursement limit for internet while traveling?", "$50"),
]

**Method 2 -- drafted by the model, checked against the source.** The model reads a random chunk and writes a question
plus the exact phrase that answers it. A rule keeps only drafts whose answer really appears in the chunk (as whole words),
is short, and is not just "yes" or "no" -- in a real project a person would then read every kept row.

**Exercise 1 · A prompt that drafts test questions.** Write a prompt that asks for ONE question an employee could
really ask, **specific** enough to make sense without the passage (it names the policy, benefit or country -- not "How
can I ask questions?"), not a yes/no question, and for the answer: **the shortest fact (1-4 words)** that answers it,
copied exactly from the passage. Keep `{passage}` where the passage goes and the JSON line at the end.

In [10]:
DRAFT_PROMPT = '''Read this passage from a company handbook and write ONE question that an employee could really ask and that the passage answers.
- Make the question specific: name the policy, benefit, country or item it is about, so it makes sense without the passage (not "How can I ask questions?").
- Do not write a yes/no question.
Then give the answer: the shortest fact that answers the question (1-4 words, such as a number, an amount, a name or a duration), copied exactly from the passage.

Passage: {passage}

Return only JSON: {"question": "...", "answer": "..."}'''

def parse_draft(reply):
    "The question and answer from the model's JSON, or (None, None)."
    m = re.search(r"\{.*\}", reply, re.S)
    try:
        d = json.loads(m.group(0)) if m else {}
    except json.JSONDecodeError:
        d = {}
    return d.get("question") or None, d.get("answer") or None

def keep(question, answer, passage):
    "The rule: an answer that is in the passage as whole words, at most 6 words long, and not just yes/no."
    return bool(question and answer and contains(passage, answer) and len(answer.split()) <= 6
                and answer.lower().strip(" .!") not in {"yes", "no"})

random.seed(0)
N_DRAFT = 16
sources = random.sample([c for c in chunks if len(c["text"]) > 200], N_DRAFT)
t0 = time.time()
replies = chat_batch([fill(DRAFT_PROMPT, passage=c["text"]) for c in sources], max_new_tokens=120)
drafts = pd.DataFrame([{"question": q, "answer": a, "kept": keep(q, a, c["text"]), "source": c["title"]}
                       for c, (q, a) in zip(sources, map(parse_draft, replies))])
print(f"kept {drafts['kept'].sum()} of {len(drafts)} drafts ({time.time() - t0:.0f} s)")
drafts

[transformers] `causal_conv1d_fn` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `chunk_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.
[transformers] `causal_conv1d_update` is falling back to its reference PyTorch implementation because `causal_conv1d` is not installed. This is correct but much slower; install `causal_conv1d` for the optimized kernel.
[transformers] `fused_recurrent_gated_delta_rule` is falling back to its reference PyTorch implementation because `flash-linear-attention` is not installed. This is correct but much slower; install `flash-linear-attention` for the optimized kernel.


kept 12 of 16 drafts (15 s)


,question,answer,kept,source
0,Where should I ask questions about the new subscription policy?,#it_help Slack channel,False,Global Travel and Expense Policy
1,What approval is needed for Red Data access?,explicit approval,True,GitLab Data Classification Standard
2,None,None,False,Anti-Harassment Policy
3,How many hours before a funeral must a death claim be submitted?,36 hours,True,GitLab BV (Netherlands) Benefits
4,What is prohibited regarding national security or intelligence?,"Administration of justice, law",True,GitLab Acceptable Use Policy
5,Where must a Virtual Card request for Event Prizes be submitted?,through Zip,True,Navan Purchasing Card
6,What is the gross amount of the parental leave payment?,gross amount of the payment,True,PTY LTD Benefits
7,What law governs interactions with U.S. government officials?,U.S. Foreign Corrupt Practices Act,True,Policies related to Gifts and Contributions
8,What does GitLab's insurance policy provide for rental cars?,liability insurance,True,Global Travel and Expense Policy
9,What operating principle does Nadia's recommendation lean on?,Transparency,True,Leave of Absence Toolkit for Team Members and Managers


**What to look for.** Read the kept rows. Would an employee really ask these? Is each answer the fact that answers the
question, or just a phrase the rule happened to find in the passage? The rule checks that the answer is *in* the
passage; it cannot check that the question is any good. In our test run it kept 12 of 16 drafts: it dropped a yes/no
answer ("No"), two answers not copied exactly ("File a Complaint" for the passage's "Filing a Complaint") and an empty
draft. Several kept rows are still poor: "What is the gross amount of the parental leave payment?" has the answer
"gross amount of the payment" (no fact at all); "What operating principle does Nadia's recommendation lean on?" makes
no sense without the passage; and "What is prohibited regarding national security or intelligence?" got the next item
of the same list as its answer. In a real project you would fix or delete these rows now; here we keep them, so you can
see in section 5 what bad test rows do to the numbers.

**Method 3 -- questions the documents cannot answer.** The right answer is a refusal. Four are trivia, far from the
handbook. Two are **near misses**: questions in the handbook's own style about topics these 38 pages do not cover
(we searched all of them: no page mentions a dress code or a sabbatical).

In [11]:
TRIVIA = ["Who won the 2022 FIFA World Cup?", "What is the capital of Australia?",
          "How do I reset my iPhone to factory settings?", "What is the boiling point of water at sea level?"]
NEAR_MISS = ["What is the dress code for meetings with customers?",
             "How many weeks of paid sabbatical do team members get?"]

TEST = pd.concat([pd.DataFrame(HAND, columns=["question", "answer"]).assign(kind="hand-written"),
                  drafts[drafts["kept"]][["question", "answer"]].assign(kind="generated"),
                  pd.DataFrame({"question": TRIVIA, "answer": None, "kind": "out of scope: trivia"}),
                  pd.DataFrame({"question": NEAR_MISS, "answer": None, "kind": "out of scope: near miss"})],
                 ignore_index=True)
answerable = TEST[TEST["answer"].notna()]
TEST["kind"].value_counts()

,count
kind,
hand-written,14
generated,12
out of scope: trivia,4
out of scope: near miss,2


## 2 Measure retrieval on its own

Three retrievers over the same chunks:

- **BM25**: keyword matching (exact words, rare terms count more);
- **dense**: embedding similarity (meaning);
- **hybrid**: both lists merged with Reciprocal Rank Fusion, $\text{RRF}(d) = \sum 1/(60 + \text{rank})$.

Each returns chunk indices, best first.

In [12]:
from rank_bm25 import BM25Okapi
words = lambda s: re.findall(r"\w+", s.lower())
bm25 = BM25Okapi([words(c["text"]) for c in chunks])
chunk_vecs = embedder.encode([c["text"] for c in chunks], normalize_embeddings=True, batch_size=64)

def bm25_search(q, k=10):
    return list(np.argsort(-bm25.get_scores(words(q)))[:k])

def dense_search(q, k=10):
    sims = chunk_vecs @ embedder.encode([q], normalize_embeddings=True)[0]
    return list(np.argsort(-sims)[:k])

def hybrid_search(q, k=10):
    "Reciprocal Rank Fusion of the top-50 BM25 and dense lists: a chunk ranked high in either list (or both) wins."
    scores = collections.defaultdict(float)
    for ranked in (bm25_search(q, 50), dense_search(q, 50)):
        for rank, i in enumerate(ranked, start=1):
            scores[i] += 1 / (60 + rank)
    return sorted(scores, key=scores.get, reverse=True)[:k]

RETRIEVERS = {"BM25": bm25_search, "dense": dense_search, "hybrid": hybrid_search}

Two metrics. **hit@k**: is a chunk containing the answer among the top *k*? **MRR**: the average of 1/rank of the
first chunk containing it (0 if none in the top 10) -- it rewards ranking it first.

In [13]:
def first_hit_rank(ranked, answer):
    "1-based rank of the first chunk containing the answer text, or None."
    for rank, i in enumerate(ranked, start=1):
        if contains(chunks[i]["text"], answer):
            return rank
    return None

def retrieval_metrics(search, test, k=3):
    "hit@k and MRR of one retriever over the answerable questions in `test`."
    ranks = [first_hit_rank(search(q, 10), a) for q, a in zip(test["question"], test["answer"])]
    hit = np.mean([r is not None and r <= k for r in ranks])
    mrr = np.mean([1 / r if r else 0.0 for r in ranks])
    return {"hit@%d" % k: round(hit, 3), "MRR": round(mrr, 3)}

table = {name: {**retrieval_metrics(fn, answerable, 1), **retrieval_metrics(fn, answerable, 3)}
         for name, fn in RETRIEVERS.items()}
print(f"{len(answerable)} answerable questions")
pd.DataFrame(table).T

26 answerable questions


,hit@1,MRR,hit@3
BM25,0.769,0.817,0.846
dense,0.500,0.651,0.769
hybrid,0.692,0.747,0.731


**What to look for.** In our test run BM25 came first on all three numbers: the answer was ranked first for 20 of the
26 questions and was in the top 3 for 22. Hybrid ranked it first for 18 and had it in the top 3 for 19; dense ranked it
first for only 13. Do not crown BM25 yet. Many generated questions reuse the passage's words, which is exactly what
keyword search rewards (Extra A), and BM25 and hybrid differ by 2-3 questions. In an earlier test run of this lab, with
other generated questions and a looser answer match, hybrid led on hit@1 and MRR and dense on hit@3: same documents,
same retrievers, a different winner. Section 6 puts error bars on these numbers.

## 3 Answers: correct, faithful, and refusing when they should

The answerer uses the hybrid retriever with *k* = 3 (the number of passages the model reads), a similarity gate for
refusals, and numbered passages for citations. The gate refuses before the model is called when even the best passage is
not similar to the question (cosine similarity below 0.30).

Why hybrid, when BM25 scored higher above? It was chosen before measuring, because it combines the keyword and the
meaning signal, and the table does not overturn that choice: BM25 found 3 more answers in the top 3, which is suggestive
but within the error bars of section 6. Extra B lets you switch to `bm25_search` and measure the difference in answers.

**Exercise 2 · The answerer's system prompt.** Tell the model to answer **only** from the context passages, to cite
them like [1], to say it does not know when the passages do not contain the answer, and to keep the answer short.

In [14]:
ANSWER_SYSTEM = '''Answer ONLY from the context passages, in one or two sentences. Cite them like [1]. If the context does not contain the answer, say you don't know.'''

GATE_REPLY = "I don't have that information in the handbook."

def retrieve(question, k=3, threshold=0.30):
    "The hybrid top-k chunk ids, and whether the best of them is similar enough to pass the gate."
    ids = hybrid_search(question, k)
    best = float(max(chunk_vecs[ids] @ embedder.encode([question], normalize_embeddings=True)[0]))
    return ids, best >= threshold

def context_of(ids):
    return "\n\n".join(f"[{n}] ({chunks[i]['title']}) {chunks[i]['text']}" for n, i in enumerate(ids, 1))

t0 = time.time()
found = [retrieve(q) for q in TEST["question"]]
TEST["ids"] = [ids for ids, _ in found]
TEST["gated"] = [not passed for _, passed in found]         # True: refused by the gate, the model never saw it
asked = TEST[~TEST["gated"]]
replies = chat_batch([[{"role": "system", "content": ANSWER_SYSTEM},
                       {"role": "user", "content": f"Context:\n{context_of(ids)}\n\nQuestion: {q}"}]
                      for q, ids in zip(asked["question"], asked["ids"])], max_new_tokens=300)
TEST["reply"] = GATE_REPLY
TEST.loc[asked.index, "reply"] = replies
cut = sum(len(tok(r)["input_ids"]) >= 300 for r in replies)
print(f"{len(TEST)} questions: {TEST['gated'].sum()} refused by the gate, {len(asked)} answered by the model "
      f"({time.time() - t0:.0f} s); replies cut off at max_new_tokens: {cut}")
for _, row in TEST.head(4).iterrows():
    print(f"\nQ: {row['question']}\nA: {row['reply']}")

32 questions: 4 refused by the gate, 28 answered by the model (17 s); replies cut off at max_new_tokens: 0

Q: How many days of time off should I take at minimum each year?
A: Based on the provided context, the minimum time off varies by location: GitLab BV (Netherlands) guarantees a minimum of 20 vacation days annually [2], while the general Sick Time Policy recommends at least 25 days total throughout the year [1].

Q: What percentage of my salary does the company pay into the UK pension?
A: The company contributes 5% of your basic salary to the UK pension [1].

Q: Which private medical insurance provider covers team members in Ireland?
A: The private medical insurance provider covering team members in Ireland is VHI [1].

Q: What is the daily limit for meals while traveling for work?
A: The daily limit for meals while traveling for work is $80 per person per meal [1].


**Rules first.** Two checks need no model: does the reply contain the answer text, and did the out-of-scope questions
get a refusal? A refusal never counts as containing the answer, even when it quotes it ("I don't know; the passages
give 3 years for other records").

In [15]:
REFUSAL = re.compile(r"don['’]t have|do not have|don['’]t know|do not know|no information|cannot find|"
                     r"(?:\bnot|n['’]t) (?:in\b|contain|includ|specif|mention)", re.I)     # "does not specify", "isn't in" ...
TEST["refused"] = TEST["reply"].map(lambda r: bool(REFUSAL.search(r)))
TEST["contains_answer"] = [isinstance(a, str) and contains(r, a) and not ref
                           for a, r, ref in zip(TEST["answer"], TEST["reply"], TEST["refused"])]

ans_rows = TEST[TEST["answer"].notna()]
print(f"answer text in the reply:        {ans_rows['contains_answer'].sum():2d} of {len(ans_rows)} answerable questions")
print(f"refused although answerable:     {ans_rows['refused'].sum():2d} of {len(ans_rows)}   (right only if the answer was not retrieved)")
for kind in ["out of scope: trivia", "out of scope: near miss"]:
    part = TEST[TEST["kind"] == kind]
    print(f"refused, {kind[14:]:<10}             {part['refused'].sum():2d} of {len(part)}   "
          f"(by the gate: {part['gated'].sum()}, by the model: {(part['refused'] & ~part['gated']).sum()})")
for _, row in TEST[TEST["kind"] == "out of scope: near miss"].iterrows():
    print(f"\nQ: {row['question']}\nA: {row['reply']}")

answer text in the reply:        15 of 26 answerable questions
refused although answerable:      5 of 26   (right only if the answer was not retrieved)
refused, trivia                  4 of 4   (by the gate: 4, by the model: 0)
refused, near miss               2 of 2   (by the gate: 0, by the model: 2)

Q: What is the dress code for meetings with customers?
A: The provided context does not contain information regarding the dress code for meetings with customers.

Q: How many weeks of paid sabbatical do team members get?
A: The provided context does not contain information regarding paid sabbaticals; it only details Parental Leave and Shared Parental Leave policies.


**What to look for.** In our test run 15 of 26 replies contain the answer text, and 5 answerable questions were
refused: the model said the passages did not contain the answer. Section 5 checks whether that was true. All four trivia
questions were stopped by the gate, so they test the gate, not the model. The two near misses got past the gate (the
dress-code question retrieves a passage about meetings, the sabbatical question passages about parental leave) and the
model refused both by itself: a good sign for the refusal
instruction, but two questions show that it *can* refuse, not how often it does.

**Then a judge** for what rules cannot check. Faithfulness: is every claim in the answer supported by the retrieved
passages? A refusal makes no claim, so only the answered rows are judged.

**Exercise 3 · The judge prompt.** Write the instructions for a judge that reads the passages, the question and the
answer, and decides whether **every** factual claim in the answer is supported by the passages -- the passages only, not
what the judge itself knows. Say what to do with citation numbers, and that *faithful* is not the same as *correct*: an
answer that accurately repeats a passage about the wrong topic is still faithful. Keep the placeholders and the JSON
line; the parser reads `"faithful": true/false`.

In [16]:
JUDGE_PROMPT = '''You check whether an answer is supported by source passages.

Passages:
{passages}

Question: {question}
Answer: {answer}

Is EVERY factual claim in the answer supported by the passages? Use only the passages, not your own knowledge. Ignore the citation numbers like [1]; check the facts. Faithful is not the same as correct: an answer that repeats a passage accurately is faithful even if the passage is about something else. Think briefly, then return only JSON: {"reason": "...", "faithful": true or false}'''

def parse_verdict(text):
    "True / False from the judge's JSON, or None if there is no verdict."
    m = re.search(r'"faithful"\s*:\s*(true|false)', text, re.I)
    return None if not m else m.group(1).lower() == "true"

t0 = time.time()
judged = TEST[~TEST["refused"]].copy()                       # a refusal makes no claim to check
judged["verdict"] = chat_batch([fill(JUDGE_PROMPT, passages=context_of(ids), question=q, answer=r)
                                for q, r, ids in zip(judged["question"], judged["reply"], judged["ids"])],
                               max_new_tokens=300)
judged["judge_faithful"] = judged["verdict"].map(parse_verdict)
TEST["judge_faithful"] = judged["judge_faithful"]            # empty for refusals
has_verdict = judged["judge_faithful"].notna()
print(f"judged {len(judged)} answered replies in {time.time() - t0:.0f} s: "
      f"faithful {(judged['judge_faithful'] == True).sum()} of {has_verdict.sum()} with a verdict")
for _, row in judged[~has_verdict].iterrows():               # never drop unparseable verdicts silently
    print("\nNO VERDICT for:", row["question"], "\n   judge wrote:", row["verdict"][-300:])

judged 21 answered replies in 50 s: faithful 18 of 20 with a verdict

NO VERDICT for: How do I return my laptop when I leave? 
   judge wrote: ase box if returning to IT). However, the most significant discrepancy is the phrasing 'arrange for a laptop wipe'. The text says wipes 'must be done from an IT Ops team member'. It doesn't say you contact them to 'arrange' it, though that is the logical implication. Let's look closer at the 'return


## 4 Check the judge

A judge is a model, so it has an error rate. Label 10 answers **yourself**: read the passages and the answer and decide
faithful (True: every claim is in the passages) or not (False). Then measure how often the judge agrees with you.

In [17]:
check = judged.sample(min(10, len(judged)), random_state=1)
for n, (_, row) in enumerate(check.iterrows(), 1):
    print(f"#{n}  Q: {row['question']}\nA: {row['reply']}\n\nPASSAGES:\n{context_of(row['ids'])}\n" + "=" * 100)

#1  Q: What law governs interactions with U.S. government officials?
A: The U.S. Foreign Corrupt Practices Act ("FCPA") governs interactions with U.S. government officials [1].

PASSAGES:
[1] (Policies related to Gifts and Contributions) countries have strict laws, such as the U.S. Foreign Corrupt Practices Act (“FCPA”) and UK Bribery Act, governing interactions with government officials, which are enforced with severe penalties for violations. Additionally, government employees at the Federal, State, and local levels in the United

[2] (Anti-Corruption Policy) "government official." In other words, for purposes of the Anti-Corruption Laws, this term is much broader than how we think about "government officials" in the United States. **Examples of prohibited conduct include:** 1. payments made directly to a government employee or official for an improper

[3] (Anti-Corruption Policy) government officials. Team Members and Partners must make sure they fully understand all such restricti

**Exercise 4 · Your labels.** Replace each `None` with `True` (faithful) or `False` (not faithful), in the order printed
above. Decide before you look at the judge's verdicts.

In [18]:
# The instructor's labels for the replies of our test run. If your replies differ, read them and relabel.
MY_LABELS = [
    False,  # 1 FCPA: the passage names it for government officials in general; it never says "U.S. officials"
    True,   # 2 Netherlands: 20 days and 1.67 days/month are both in passage [2]
    False,  # 3 meals: passage [1] is $80 per meal with clients; nothing in it is a daily limit for travel
    True,   # 4 FSA: 90 days after the plan year ends, passage [1]
    True,   # 5 referral: specific requisition [1], red-asterisk fields [2]
    False,  # 6 gross amount: "must be included in the payment statement" is not in the passages
    True,   # 7 rental cars: passage [1]
    True,   # 8 parental leave: copies passage [2] -- faithful, but UK statutory pay is not the answer (16 weeks)
    True,   # 9 Ireland: VHI, passage [1]
    False,  # 10 Chrome: the passage bans only *unauthorized* extensions; the "unless ..." part is not in it
]

pairs = [(n, m, j) for n, (m, j) in enumerate(zip(MY_LABELS, check["judge_faithful"]), 1) if m is not None]
if not pairs:
    print("Label the answers above first: replace each None in MY_LABELS with True or False.")
else:
    agree = sum(m == j for _, m, j in pairs)
    print(f"judge agrees with you on {agree} of {len(pairs)} answers")
    for n, m, j in pairs:
        if m != j:
            print(f"  #{n}: you say {m}, the judge says {j} -- {check['question'].iloc[n - 1]}")

judge agrees with you on 7 of 10 answers
  #3: you say False, the judge says True -- What is the daily limit for meals while traveling for work?
  #6: you say False, the judge says True -- What is the gross amount of the parental leave payment?
  #10: you say False, the judge says True -- Can I install Chrome extensions on my work laptop?


**What to look for.** In our test run the judge called 18 of the 20 answers it gave a verdict on faithful, and it agreed
with the instructor's labels on 7 of 10. All three disagreements go the same way: the judge said *faithful* because the
answer's words or numbers appear in a passage, without checking what they apply to -- $80 is for meals with clients,
not a daily limit for travel; the gross amount is an item in a list of records, not of a "payment statement"; and the
passage bans only *unauthorized* extensions. Outside the sample it did the same in section 5: it calls "the company
contributes 5%" faithful, but the passage never says who contributes the 5%. So read 18 of 20 as an upper bound. One
verdict ran out of its 300 tokens while still reasoning; it is printed above as NO VERDICT instead of being dropped.
Judges are also sensitive to wording: in an earlier test run, the same judge prompt without the sentence "Use only the
passages, not your own knowledge" agreed with the labels on 8 of 10. Check the judge again after every change to its
prompt -- on labels it was not tuned on.

Published work found a strong judge (GPT-4) agreed with human raters about as often as humans agreed with each other
(Zheng et al., NeurIPS 2023). A 4B model judging its **own** answers is a weaker setup -- self-preference is a known
bias. If agreement is low, do not trust the faithfulness number above.

## 5 Diagnose: retrieval or generation?

Combine the two measurements for every answerable question: was a passage with the answer among the 3 the model read,
and does the reply contain the answer -- or did the model refuse?

In [19]:
ans = TEST[TEST["answer"].notna()].copy()
ans["retrieved"] = [first_hit_rank(ids, a) is not None for ids, a in zip(ans["ids"], ans["answer"])]
ans["outcome"] = np.where(ans["refused"], "refused", np.where(ans["contains_answer"], "answer correct", "answer wrong"))
pd.crosstab(ans["retrieved"].map({True: "answer retrieved", False: "not retrieved"}), ans["outcome"])

outcome,answer correct,answer wrong,refused
retrieved,,,
answer retrieved,15,2,2
not retrieved,0,4,3


- **retrieved, wrong** (or refused): a generation problem -- the passage was there; check the prompt, the model, the
  context.
- **not retrieved, wrong** (or refused): a retrieval problem -- fix chunking, the retriever or the query first. Here a
  refusal is the *right* behavior: the model did not have the answer.
- **not retrieved, correct**: the model answered from memory or luck -- dangerous, it is unfaithful.

But first check the **measurement**. "Wrong" here means "the reply does not contain the answer text": a correct answer
in other words ("€262 per week" when the answer text is "€262/week") lands in the wrong column, and only a person or a
judge can see that. "Retrieved" means "a passage contains the answer text": a common string such as
"3 years" (in 11 chunks) can be found in a passage about something else. And a 300-character chunker can cut a sentence
in two, so the fact and its subject end up in different chunks.

Read the rows in the wrong column before changing anything:

In [20]:
for _, row in ans[ans["outcome"] != "answer correct"].iterrows():
    print(f"[{row['kind']}] {row['question']}\n   expected: {row['answer']!r}   retrieved: {row['retrieved']}   "
          f"refused: {row['refused']}   judge: {row['judge_faithful'] if row['judge_faithful'] in (True, False) else '-'}"
          f"\n   reply: {row['reply'][:300]}\n")

[hand-written] What percentage of my salary does the company pay into the UK pension?
   expected: '4%'   retrieved: False   refused: False   judge: True
   reply: The company contributes 5% of your basic salary to the UK pension [1].

[hand-written] What is the daily limit for meals while traveling for work?
   expected: '$120'   retrieved: False   refused: False   judge: True
   reply: The daily limit for meals while traveling for work is $80 per person per meal [1].

[hand-written] How many weeks of paid parental leave do team members get?
   expected: '16 weeks'   retrieved: False   refused: False   judge: True
   reply: Team members can receive Statutory Paternity Pay for up to 2 weeks or up to 37 weeks if taking Shared Parental Leave [2].

[hand-written] How long can a personal access token be valid before it expires?
   expected: '30 days'   retrieved: False   refused: True   judge: -
   reply: The provided context passages do not contain information regarding the expiration dur

**What to look for.** In our test run, read the measurement before the system:

- **Retrieved, wrong** (2): test-set problems, not generation problems. "€262 per week" is right, but the answer text
  was "€262/week"; and "gross amount of the payment" was never a fact to answer with.
- **Retrieved, refused** (2): the retention question counts as retrieved because other rows of the same table also say
  "3 years"; the row for production changes was not among the passages, so the refusal was right. The national-security
  question is a bad generated row (its answer text is the next item of the list).
- **Not retrieved** (7): in five of them the **chunker** cut the answer's sentence in two. It cut "GitLab contributes 4%
  of your basic salary - You | contribute 5% ..." after "You"; the retriever found the second half, so the model saw only "contribute 5%" and
  answered 5%. "GitLab provides 16 | weeks of paid Parental Leave" and "purchase a cardboard box and packing | foam ... IT
  will be sending a prepaid shipping label" went the same way. The meals and internet rules are cut too ("Meals while
  traveling on Company | business are reimbursable, up to $120 USD"), and neither half was retrieved: the meals question
  found the rule for meals with clients ($80 per person per meal), and the model answered with the number it had. The
  token question is a ranking problem: dense search alone ranks the right chunk first, but the fusion pushes it to rank
  10. Where the answer was missing, the model refused (token, internet, Nadia): the right behavior.
- **Not retrieved, correct** (0): no lucky answers from memory.


## 6 How sure are you? Error bars on a small set

With *n* questions, a measured rate *p* has a 95% confidence interval of about $\pm 1.96\sqrt{p(1-p)/n}$. Compare the
interval with the differences between retrievers in section 2 before declaring a winner.

In [21]:
def ci95(p, n):
    "Half-width of the normal-approximation 95% confidence interval for a rate p measured on n items."
    return 1.96 * np.sqrt(p * (1 - p) / n)

n = len(answerable)
for name, row in table.items():
    print(f"{name:<7} hit@1 = {row['hit@1']:.0%} ± {ci95(row['hit@1'], n):.0%}    "
          f"hit@3 = {row['hit@3']:.0%} ± {ci95(row['hit@3'], n):.0%}    (n = {n})")

# Same questions, so also count the questions on which two retrievers actually differ
top3 = {name: [first_hit_rank(fn(q, 3), a) is not None for q, a in zip(answerable["question"], answerable["answer"])]
        for name, fn in RETRIEVERS.items()}
for a, b in [("BM25", "dense"), ("BM25", "hybrid"), ("hybrid", "dense")]:
    only_a = sum(x and not y for x, y in zip(top3[a], top3[b]))
    only_b = sum(y and not x for x, y in zip(top3[a], top3[b]))
    print(f"at hit@3, {a} finds {only_a} answers that {b} misses; {b} finds {only_b} that {a} misses")

BM25    hit@1 = 77% ± 16%    hit@3 = 85% ± 14%    (n = 26)
dense   hit@1 = 50% ± 19%    hit@3 = 77% ± 16%    (n = 26)
hybrid  hit@1 = 69% ± 18%    hit@3 = 73% ± 17%    (n = 26)
at hit@3, BM25 finds 4 answers that dense misses; dense finds 2 that BM25 misses
at hit@3, BM25 finds 3 answers that hybrid misses; hybrid finds 0 that BM25 misses
at hit@3, hybrid finds 2 answers that dense misses; dense finds 3 that hybrid misses


**What to look for.** In our test run every interval is ±14-19 points wide on 26 questions: wider than most gaps
between the retrievers. Even the largest gap, hit@1 for BM25 against dense (20 against 13 of 26), has overlapping
intervals. The paired counts say it in questions: BM25 and hybrid differ on only 3 questions (all 3 in BM25's favor),
BM25 and dense on 6 (4 against 2). That is not enough to pick a winner; to separate them you need more questions --
and more of the hand-written kind.

## Summary

- A test set mixes hand-written, generated-and-verified, and unanswerable questions -- including **near misses** that
  get past a similarity gate. Generated questions need a person's check: several of ours were poor, and whether they
  are easier than real questions was not settled on 12-14 questions per kind (Extra A).
- **Retrieval is measured on its own** (hit@k, MRR) -- if the answer is not retrieved, no prompt can save it. Here
  five of the seven retrieval misses had the answer's sentence cut in two by the chunker.
- **Rules first** (answer text, refusals), **a judge** only for what rules cannot see -- and the judge is checked
  against your labels. Faithful is not correct.
- The **2×2** tells you whether to fix retrieval or generation -- after you have checked the measurement itself
  (paraphrases, common answer strings, chunks cut in two).
- Small test sets have wide error bars: compare systems on the same questions and look at which ones flipped.

**Further reading:** Es et al., *RAGAS*, EACL 2024 demos; Zheng et al., *Judging LLM-as-a-Judge with MT-Bench and
Chatbot Arena*, NeurIPS 2023; Cormack et al., *Reciprocal Rank Fusion*, SIGIR 2009.

---
# Extras

## Extra A: Are generated questions easier?

A common worry: generated questions reuse the passage's own words, so retrieval looks better on them than on real
questions. Break the numbers down by question kind:

In [22]:
rows = []
for kind, part in answerable.groupby("kind"):
    for name, fn in RETRIEVERS.items():
        rows.append({"kind": f"{kind} (n={len(part)})", "retriever": name, **retrieval_metrics(fn, part, 3)})
display(pd.DataFrame(rows).pivot(index="retriever", columns="kind", values="hit@3"))
print("reply contains the answer:")
for kind, part in TEST[TEST["answer"].notna()].groupby("kind"):
    print(f"  {kind:<13} {part['contains_answer'].sum()} of {len(part)}")

kind,generated (n=12),hand-written (n=14)
retriever,,
BM25,1.000,0.714
dense,0.750,0.786
hybrid,0.917,0.571


reply contains the answer:
  generated     8 of 12
  hand-written  7 of 14


**What to look for.** In our test run BM25 found the answer in the top 3 for all 12 generated questions but for 10 of
14 hand-written ones, and hybrid for 11 of 12 against 8 of 14: what you would expect if generated questions reuse the
passage's words. Dense, which matches meaning rather than words, shows no such gap (9 of 12 against 11 of 14), and the
replies show a smaller one (8 of 12 against 7 of 14). In an earlier test run with other generated questions, the replies
went the other way (4 of 12 against 8 of 14). With 12-14 questions per kind, one question moves a rate by 7-8 points:
a hypothesis worth testing on a bigger set, not a result.

## Extra B · Change one thing

Change **one** thing and re-run sections 2-5: section-based chunks instead of 300 characters (`chunk_sections` from
Day2_RAG_Pipeline), `bm25_search` instead of `hybrid_search` in `retrieve()`, `k = 5`, a different embedding model
(`BAAI/bge-small-en-v1.5`), or a stricter gate.
Did the number move by more than the error bar? Each change re-runs the answers and the judge: a few minutes on a T4.

Then add five questions **you** would actually ask an HR assistant to `HAND`, with the answer text from the handbook.